In [ ]:
# ==========================================
# 1. IMPORTS & HELPER FUNCTIONS
# ==========================================
import json
import os
import re
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime

def add_subfield(parent, code, text):
    """Helper function to add a MARC subfield element."""
    subfield = ET.SubElement(parent, "subfield", {"code": code})
    subfield.text = str(text)
    return subfield

def generate_gnd_marcxml_from_json(json_input_path: str, mapping_path: str, output_folder: str):
    """Converts corporate authority records from JSON to GND-compliant MARC21 XML files."""
    os.makedirs(output_folder, exist_ok=True)

    # Load input JSON records
    with open(json_input_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    # Load classification mapping
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping = json.load(f)

    klassisch_ids = set(mapping.get("ids_klassisch", []))
    today_str = datetime.now().strftime("%d.%m.%Y")
    stand_text = f"Stand: {today_str}"

    # Regex matching corporate legal forms at the end of canonical names
    rechtsform_pattern = re.compile(
        r"[\s,]+(AG|Aktiengesellschaft|Ltd\.?|Limited|Sàrl|Société à responsabilité limitée)$",
        re.IGNORECASE
    )

    print(f"Starting MARCXML generation in destination: '{output_folder}'...")

    for record in records:
        record_id = record.get("id", "unknown")
        canonical_name = record.get("canonical", "").strip()
        is_klassisch = record_id in klassisch_ids
        hat_erlaubte_rechtsform = bool(rechtsform_pattern.search(canonical_name))

        name_110 = canonical_name
        erzeuge_410_kopie = False

        # Rule: Strip legal form for 110 if classic classification matches
        if is_klassisch and canonical_name and hat_erlaubte_rechtsform:
            erzeuge_410_kopie = True
            name_110 = rechtsform_pattern.sub("", canonical_name).strip()

        # Create MARC21 record root
        record_elem = ET.Element("record", {
            "type": "Authority",
            "xmlns": "http://www.loc.gov/MARC21/slim",
            "xmlns:xsi": "http://www.w3.org/2001/XMLSchema-instance",
            "xsi:schemaLocation": "http://www.loc.gov/MARC21/slim http://www.loc.gov/standards/marcxml/schema/MARC21slim.xsd"
        })

        # 1. Leader
        leader = ET.SubElement(record_elem, "leader")
        leader.text = "00000nz  a2200000nc 4500"

        # 2. Field 075: Entity Type (Corporate Body -> kif)
        df_075 = ET.SubElement(record_elem, "datafield", {"tag": "075", "ind1": " ", "ind2": " "})
        add_subfield(df_075, "b", "kif")
        add_subfield(df_075, "2", "gndspec")

        # 3. Field 043: Country Code
        meta = record.get("meta", {})
        nationality = meta.get("nationality", "")
        if nationality and nationality != "??":
            df_043 = ET.SubElement(record_elem, "datafield", {"tag": "043", "ind1": " ", "ind2": " "})
            add_subfield(df_043, "c", f"XA-{nationality}")

        # 4. Field 110: Main Entry (Heading)
        if name_110:
            df_110 = ET.SubElement(record_elem, "datafield", {"tag": "110", "ind1": "2", "ind2": " "})
            add_subfield(df_110, "a", name_110)

        # 5. Field 410: See-From Tracing (Aliases)
        for alias in record.get("aliases", []):
            if alias:
                df_410 = ET.SubElement(record_elem, "datafield", {"tag": "410", "ind1": "2", "ind2": " "})
                add_subfield(df_410, "a", alias)

        # 410 Exact copy with $g nauv when legal form was stripped from 110
        if erzeuge_410_kopie:
            df_410_exact = ET.SubElement(record_elem, "datafield", {"tag": "410", "ind1": "2", "ind2": " "})
            add_subfield(df_410_exact, "a", canonical_name)
            add_subfield(df_410_exact, "g", "nauv")

        # 6. Field 548: Date of Establishment/Dissolution
        inception = meta.get("inception", "")
        dissolved = meta.get("dissolved", "")
        if inception or dissolved:
            df_548 = ET.SubElement(record_elem, "datafield", {"tag": "548", "ind1": " ", "ind2": " "})
            date_str = f"{inception}-{dissolved}" if dissolved else f"{inception}-"
            add_subfield(df_548, "a", date_str)
            add_subfield(df_548, "4", "datb")
            add_subfield(df_548, "4", "https://d-nb.info/standards/elementset/gnd#dateOfEstablishment")
            add_subfield(df_548, "w", "r")
            add_subfield(df_548, "i", "Zeitraum")

        # 7. Field 670: Source Data / Web Links
        for link in meta.get("links", []):
            if link:
                df_670 = ET.SubElement(record_elem, "datafield", {"tag": "670", "ind1": " ", "ind2": " "})
                add_subfield(df_670, "a", "Homepage")
                add_subfield(df_670, "b", stand_text)
                add_subfield(df_670, "u", link)

        # Pretty print XML
        xml_string = ET.tostring(record_elem, encoding="utf-8")
        reparsed = minidom.parseString(xml_string)
        pretty_xml = reparsed.toprettyxml(indent="  ", encoding="utf-8")

        # Write file
        file_output_path = os.path.join(output_folder, f"{record_id}.xml")
        with open(file_output_path, "wb") as f:
            f.write(pretty_xml)

    print(f"Success! MARCXML authority records written to '{output_folder}'.")


# ==========================================
# 2. CONFIGURATION & EXECUTION
# ==========================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else "."
DATA_DIR = os.path.join(BASE_DIR, "data")

JSON_INPUT = os.path.join(DATA_DIR, "companies_test_50.json")
MAPPING_INPUT = os.path.join(DATA_DIR, "classification_mapping.json")
OUTPUT_DIR = os.path.join(DATA_DIR, "Test_Hauptansetzung")

if os.path.exists(JSON_INPUT) and os.path.exists(MAPPING_INPUT):
    generate_gnd_marcxml_from_json(JSON_INPUT, MAPPING_INPUT, OUTPUT_DIR)
else:
    print("[-] Input files not found. Update DATA_DIR path before running.")
